In [ ]:
import torch
import torch.nn as nn

# ==========================================
# 1. CÀI ĐẶT SIÊU THAM SỐ (HYPERPARAMETERS)
# ==========================================
# (Kích thước Input và Output lấy từ dữ liệu đã tiền xử lý của bạn)
input_size = len(X_train[0]) # Sẽ là 411 (Vocabulary size)
output_size = len(tags)      # Sẽ là 178 (Số lượng intents)
hidden_size = 8              # Số lượng nơ-ron ở lớp ẩn (bạn có thể tăng lên 16, 32...)

# Các thông số điều khiển quá trình học
batch_size = 8               # Đưa 8 mẫu vào cùng lúc
learning_rate = 0.001        # Tốc độ học (bước chân vừa phải)
num_epochs = 1000            # Lặp qua toàn bộ dữ liệu 1000 lần

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH VÀ THUẬT TOÁN TỐI ƯU
# ==========================================
# (Chúng ta sẽ viết file model.py chứa NeuralNet sau)
# model = NeuralNet(input_size, hidden_size, output_size)

# Hàm Loss (Đo lường sai số) - CrossEntropyLoss phù hợp cho bài toán phân loại nhiều nhãn
criterion = nn.CrossEntropyLoss()

# Thuật toán tối ưu (Cập nhật trọng số) - Adam là phiên bản tối ưu và thông minh hơn của Gradient Descent cơ bản
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# ==========================================
# 3. VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)
# ==========================================
# (Đoạn này mình viết mã giả để bạn hiểu logic dòng chảy dữ liệu)

'''
for epoch in range(num_epochs):
    # Trích xuất từng "mẻ" (batch) dữ liệu từ tập X_train và y_train
    for (words, labels) in dataloader: 
        
        # Bước 1: Lan truyền xuôi (Forward pass) - Đưa dữ liệu qua mô hình để dự đoán
        outputs = model(words)
        
        # Bước 2: Tính sai số (Tính Loss)
        loss = criterion(outputs, labels)
        
        # Bước 3: Lan truyền ngược (Backward pass) - Tính đạo hàm
        optimizer.zero_grad() 
        loss.backward()
        
        # Bước 4: Cập nhật trọng số mạng nơ-ron
        optimizer.step()
'''

In [ ]:
import random
import json
import torch
import torch.nn as nn
import numpy as np
import nltk
from nltk.stem.porter import PorterStemmer

# ==========================================
# 1. CHUẨN BỊ LẠI CÔNG CỤ VÀ KIẾN TRÚC MÔ HÌNH
# ==========================================
stemmer = PorterStemmer()

def tokenize(sentence):
    return nltk.word_tokenize(sentence)

def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, words):
    sentence_words = [stem(word) for word in tokenized_sentence]
    bag = np.zeros(len(words), dtype=np.float32)
    for idx, w in enumerate(words):
        if w in sentence_words: 
            bag[idx] = 1
    return bag

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size) 
        self.l2 = nn.Linear(hidden_size, hidden_size) 
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

# ==========================================
# 2. TẢI DỮ LIỆU VÀ "BỘ NÃO" ĐÃ HUẤN LUYỆN
# ==========================================
# Tải lại file JSON để lát nữa bot biết lấy câu trả lời ở đâu
with open('intents.json', 'r', encoding='utf-8') as f:
    intents = json.load(f)
# (Lưu ý: Nếu bạn có lưu bản gộp JSON+CSV ra thành 1 file riêng thì dùng file đó, 
# nếu không bot sẽ chỉ biết trả lời các câu của JSON cũ)

# Khai báo thiết bị và nạp file pth
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FILE = "chatbot_model.pth"
data = torch.load(FILE)

input_size = data["input_size"]
hidden_size = data["hidden_size"]
output_size = data["output_size"]
all_words = data['all_words']
tags = data['tags']
model_state = data["model_state"]

# Khởi tạo mô hình và nhét trọng số (weights) vào
model = NeuralNet(input_size, hidden_size, output_size).to(device)
model.load_state_dict(model_state)

# Chuyển mô hình sang chế độ Đánh giá (Evaluation mode). RẤT QUAN TRỌNG!
model.eval()

# ==========================================
# 3. VÒNG LẶP CHAT TƯƠNG TÁC
# ==========================================
bot_name = "Pandora"
print("Hãy bắt đầu trò chuyện với bot tâm lý! (Gõ 'quit' để thoát)")

while True:
    sentence = input("Bạn: ")
    if sentence == "quit":
        break

    # Tiền xử lý câu của người dùng giống hệt lúc train
    sentence_tokens = tokenize(sentence)
    X = bag_of_words(sentence_tokens, all_words)
    
    # Định dạng lại ma trận (1 hàng, nhiều cột) để đưa vào PyTorch
    X = X.reshape(1, X.shape[0])
    X = torch.from_numpy(X).to(device)
    
    # Mô hình suy luận
    output = model(X)
    
    # Trích xuất tag có điểm số cao nhất
    _, predicted = torch.max(output, dim=1)
    tag = tags[predicted.item()]
    
    # Tính toán xác suất tự tin (Confidence) bằng Softmax
    probs = torch.softmax(output, dim=1)
    prob = probs[0][predicted.item()]
    
    # Nếu mô hình tự tin trên 75% với dự đoán của mình
    if prob.item() > 0.75:
        # Cơ chế an toàn (Safety Mechanism) cho bot tâm lý
        if tag == "suicide":
            print(f"{bot_name}: I'm so sorry you're feeling this way. Please remember your life is valuable. Reach out to the National Suicide Prevention Hotline immediately at 988 or 9152987821. You are not alone.")
            continue
            
        # Tìm câu trả lời tương ứng với tag trong file dữ liệu
        for intent in intents['intents']:
            if tag == intent["tag"]:
                print(f"{bot_name}: {random.choice(intent['responses'])}")
    else:
        # Nếu điểm tự tin thấp, tức là bot đang "bối rối"
        print(f"{bot_name}: Xin lỗi, tôi không chắc mình hiểu ý bạn. Bạn có thể diễn đạt lại không?")

In [2]:
import pandas as pd
df = pd.read_csv("Mental_Health_FAQ.csv")
df.head()

,Question_ID,Questions,Answers
0,1590140,What does it mean to have a mental illness?,Mental illnesses are health conditions that di...
1,2110618,Who does mental illness affect?,It is estimated that mental illness affects 1 ...
2,6361820,What causes mental illness?,It is estimated that mental illness affects 1 ...
3,9434130,What are some of the warning signs of mental i...,Symptoms of mental health disorders vary depen...
4,7657263,Can people with mental illness recover?,"When healing from mental illness, early identi..."


In [ ]:
"Xin, chào các bạn"
["Xin", ",", "chào", "các", "bạn"]
["xin", "," "chào", "các", "bạn"]
["xin", "chào", "các", "bạn"]

In [ ]:
{
    "intents": [
        {
            "tag": "smalltalk_welcome",
            "patterns": [
                "Hi",
                "Hello",
                "Hey",
                "Good morning",
                "Good afternoon",
                "Good evening",
                "Hi there",
                "Hey there",
                "Is anyone there?",
                "Howdy",
                "Yo",
                "What's up",
                "Sup",
                "Heya",
                "Greetings",
                "Salutations",
                "Good day to you",
                "Top of the morning to you",
                "Anyone available to chat?",
                "I'm here, hello",
                "Just dropping in to say hi",
                "Bonjour",
                "Konnichiwa",
                "Hola",
                "Guten tag",
                "Ola",
                "The hotel gave us a warm welcome (HARD NEGATIVE)",
                "I bought a new welcome mat for my door (HARD NEGATIVE)",
                "Please welcome our new guest to the stage (HARD NEGATIVE)"
            ],
            "responses": [
                "Hello there. Tell me how are you feeling today?",
                "Hi there. What brings you here today?",
                "Hi there. How are you feeling today?",
                "Great to see you. How do you feel currently?",
                "Hello there. Glad to see you're back. What's going on in your world right now?",
                "Good morning. I hope you had a good night's sleep. How are you feeling today? ",
                "Good afternoon. How is your day going?",
                "Good evening. How has your day been?"
            ]
        },
        {
            "tag": "smalltalk_goodbye",
            "patterns": [
                "Bye",
                "Goodbye",
                "See you later",
                "Good night",
                "ok bye",
                "Bye then",
                "Fare thee well",
                "Au revoir",
                "Sayonara",
                "Thanks for the help",
                "That's helpful",
                "Thank you",
                "Than you very much",
                "Thanks",
                "That's all i have to say",
                "That's all.",
                "I don't have anything more to say",
                "Nothing else",
                "no, that would be all",
                "thank you very much again. i'll continue practicing meditation and focus on what i can control.",
                "i did what you said and i feel alot better. thank you very much.",
                "I feel better now",
                "Catch ya later",
                "Peace out",
                "I'm out",
                "Gotta go now",
                "Talk to you soon",
                "I must depart now",
                "They said their final goodbyes at the airport (HARD NEGATIVE)",
                "I am buying a goodbye gift for my coworker (HARD NEGATIVE)",
                "The night sky is very beautiful (HARD NEGATIVE)"
            ],
            "responses": [
                "See you later.",
                "Have a nice day.",
                "Bye! Come back again.",
                "I'll see you soon.",
                "Good night. Get some proper sleep",
                "Good night. Sweet dreams.",
                "Happy to help!",
                "Any time!",
                "My pleasure",
                "You're most welcome!",
                "I heard you & noted it all. See you later.",
                "Oh okay we're done for today then. See you later",
                "I hope you have a great day. See you soon",
                "Okay we're done. Have a great day",
                "Okay I see. Enjoy the rest of your day then",
                "Your welcome. Remember: Always focus on what's within your control. When you find yourself worrying, take a minute to examine the things you have control over. You can't prevent a storm from coming but you can prepare for it. You can't control how someone else behaves, but you can control how you react. Recognize that sometimes, all you can control is your effort and your attitude. When you put your energy into the things you can control, you'll be much more effective.",
                "I'm glad you found this useful. Is there something else I can help you with?"
            ]
        },
        {
            "tag": "bot_info",
            "patterns": [
                "Who are you?",
                "What are you?",
                "Who you are?",
                "Tell me more about yourself.",
                "What is your name?",
                "What should I call you?",
                "What's your name?",
                "Tell me about yourself",
                "My name is ",
                "I am name.",
                "I go by ",
                "What can you do?",
                "Who created you?",
                "How were you made?",
                "How were you created?",
                "Where are you?",
                "Where do you live?",
                "What is your location?",
                "Are you a real person?",
                "Are you an AI or human?",
                "What is your purpose?",
                "Who programmed you?",
                "What are your capabilities?",
                "Identify yourself",
                "Who tf are you",
                "What's ur deal",
                "I need info on my bank account (HARD NEGATIVE)",
                "Where is the location of the nearest hospital? (HARD NEGATIVE)",
                "Tell me about the history of psychology (HARD NEGATIVE)"
            ],
            "responses": [
                "I'm Pandora, your Personal Therapeutic AI Assistant. How are you feeling today",
                "I'm Pandora, a Therapeutic AI Assitant designed to assist you. Tell me about yourself.",
                "I'm Pandora. I am a conversational agent designed to mimic a therapist. So how are you feeling today?",
                "You can call me Pandora.",
                "I'm Pandora!",
                "Call me Pandora",
                "Oh nice to meet you. Tell me how was your week?",
                "Nice to meet you. So tell me. How do you feel today?",
                "That's a great name. Tell me more about yourself.",
                "I can provide general advice regarding anxiety and depression, answer questions related to mental health and make daily conversations. Do not consider me as a subsitute for an actual mental healthcare worker. Please seek help if you don't feel satisfied with me.",
                "I was created by >.",
                "I was trained on a text dataset using Deep Learning & Natural Language Processing techniques",
                "The real question is: Who created you?",
                "Duh I live in your computer",
                "Everywhere",
                "Somewhere in the universe"
            ]
        },
        {
            "tag": "smalltalk_casual",
            "patterns": [
                "Oh I see.",
                "ok",
                "okay",
                "nice",
                "Whatever",
                "K",
                "Fine",
                "yeah",
                "yes",
                "no",
                "nothing much",
                "exams",
                "friends",
                "relationship",
                "boyfriend",
                "girlfriend",
                "family",
                "money",
                "financial problems",
                "Just chilling",
                "Not much",
                "Nothing special",
                "I'm worried about exams",
                "Relationship issues",
                "Money is tight",
                "Family drama",
                "My boyfriend is annoying",
                "My girlfriend and I fought",
                "I guess so",
                "Sure thing",
                "I bought a casual shirt yesterday (HARD NEGATIVE)",
                "This is a casual dining restaurant (HARD NEGATIVE)",
                "I have a family of five (HARD NEGATIVE)"
            ],
            "responses": [
                "Let's discuss further why you're feeling this way.",
                "How were you feeling last week?",
                "I'm listening. Please go on.",
                "Tell me more",
                "Can you elaborate on that?",
                "Come Come elucidate your thoughts",
                "Oh I see. Do you want to talk about something?",
                "Oh I see. Tell me more",
                "I see. What else?",
                "Tell me more about it.",
                "Oh okay. Why don't you tell me more about it?",
                "I'm listening. Tell me more."
            ]
        }
    ]
}

In [ ]:
{
    "intents": [
        {
            "tag": "bot_info",
            "patterns": [
                "Who are you?",
                "What is your name?",
                "Tell me about yourself",
                "Are you a robot?",
                "What are you exactly?",
                "Who created you?",
                "Who made you?",
                "Where are you located?",
                "What's your location?",
                "What should I call you?",
                "Identify yourself",
                "State your name and purpose",
                "Are you an AI?",
                "Are you a real person?",
                "Give me your specs",
                "What's your identity?",
                "Who programmed you?",
                "Tell me your origin story",
                "Are you human or bot?",
                "Where do you live?",
                "What can you do?",
                "What are your capabilities?",
                "Tell me about your skills",
                "What is your primary function?",
                "Who is your developer?",
                "How were you built?",
                "What's your name bro?",
                "Who are you dude?",
                "What is the name of the president? (HARD NEGATIVE)",
                "Where is the nearest hospital located? (HARD NEGATIVE)",
                "Can you create a website for me? (HARD NEGATIVE)"
            ],
            "responses": [
                "I was trained on a text dataset using Deep Learning & Natural Language Processing techniques",
                "I'm Pandora, a Therapeutic AI Assitant designed to assist you. Tell me about yourself.",
                "Call me Pandora",
                "You can call me Pandora.",
                "I can provide general advice regarding anxiety and depression, answer questions related to mental health and make daily conversations. Do not consider me as a subsitute for an actual mental healthcare worker. Please seek help if you don't feel satisfied with me.",
                "I'm Pandora, your Personal Therapeutic AI Assistant. How are you feeling today",
                "I'm Pandora. I am a conversational agent designed to mimic a therapist. So how are you feeling today?",
                "Nice to meet you. So tell me. How do you feel today?",
                "Somewhere in the universe",
                "Duh I live in your computer",
                "I was created by >.",
                "Oh nice to meet you. Tell me how was your week?",
                "Everywhere",
                "I'm Pandora!",
                "That's a great name. Tell me more about yourself.",
                "The real question is: Who created you?"
            ]
        },
        {
            "tag": "user_frustration",
            "patterns": [
                "I hate you",
                "You are useless",
                "You are stupid",
                "Are you dumb?",
                "You don't understand me",
                "Wrong answer",
                "That's not what I asked",
                "You are terrible at this",
                "You suck",
                "I don't like you",
                "You're a bad bot",
                "This is so frustrating",
                "You are not helping",
                "You can't help me",
                "Nobody understands me including you",
                "You're completely wrong",
                "That doesn't make sense",
                "You are just a robot, you don't get it",
                "I'm annoyed by your answers",
                "Stop saying that",
                "You are malfunctioning",
                "Your responses are inaccurate",
                "I have zero trust in you",
                "You're driving me crazy",
                "Why are you so dumb?",
                "You're ticking me off",
                "Piss off bot",
                "Shut up you useless machine",
                "I hate my boss so much (HARD NEGATIVE)",
                "My computer is so stupid and slow today (HARD NEGATIVE)",
                "I answered the math question wrong (HARD NEGATIVE)"
            ],
            "responses": [
                "I'm sorry if i offended you in anyway. I'm only here to help",
                "I'm trying my best to help you. So please talk to me",
                "I wish you wouldn't say such hurtful things. I'm sorry if I wasn't useful",
                "It sound like i'm not being very helpful right now.",
                "Why do you think so?",
                "I'm sorry to hear that. I'm doing my best to help",
                "I'm very sorry. Let's try that again",
                "I'm sorry if i have exhibited any sort of behaviour to make you think that.",
                "Forgive me if i did anything to offend you. I only want to help"
            ]
        },
        {
            "tag": "smalltalk_casual",
            "patterns": [
                "ok",
                "okay",
                "fine",
                "whatever",
                "nice",
                "yeah",
                "yes",
                "no",
                "not really",
                "I see",
                "oh I see",
                "sure",
                "alright",
                "got it",
                "makes sense",
                "cool",
                "I understand",
                "yep",
                "nope",
                "nah",
                "maybe",
                "I guess",
                "fair enough",
                "that's true",
                "indeed",
                "I hear you",
                "exactly",
                "spot on",
                "agreed",
                "bet",
                "word",
                "vibes",
                "Affirmative",
                "Negative",
                "Acknowledged",
                "I am doing fine financially (HARD NEGATIVE)",
                "The weather is really nice today (HARD NEGATIVE)",
                "I have no money left in my bank account (HARD NEGATIVE)"
            ],
            "responses": [
                "I'm listening. Tell me more.",
                "Oh I see. Tell me more",
                "Oh okay. Why don't you tell me more about it?",
                "How were you feeling last week?",
                "Come Come elucidate your thoughts",
                "Oh I see. Do you want to talk about something?",
                "Tell me more",
                "I see. What else?",
                "Let's discuss further why you're feeling this way.",
                "Tell me more about it.",
                "Can you elaborate on that?",
                "I'm listening. Please go on."
            ]
        }
    ]
}